## Simulation pipeline

### Simulation data generation

In [1]:
import os, glob
import numpy as np
import pandas as pd
import nibabel as nib
import torch

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

MASK_PATH = "../data/MNI152_T1_2mm_brain.nii.gz"

output_dir = "../examples/simulation"
os.makedirs(output_dir, exist_ok=True)
voxel_dir = os.path.join(output_dir, "voxel_data")
os.makedirs(voxel_dir, exist_ok=True)
subj_dir = os.path.join(voxel_dir, "subjects")
os.makedirs(subj_dir, exist_ok=True)

In [2]:
# ============================================================
# Load mask
# ============================================================
info1 = nib.load(MASK_PATH)
mask_data = info1.get_fdata()
mask_bool = mask_data > 0
mask_affine = info1.affine
mask_header = info1.header

In [3]:
# ============================================================
# Simulation parameters
# ============================================================
n_sample = 500
n_snps = 100     
causal_snp_idx = 50   # selected snp adding gaussian signals
center_voxel = (44, 90, 29)
mask_rate = 0.5
sigma = (1.5, 5, 1.5)
amplitude = 1.0
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [4]:
# ============================================================
# Generate SNPs
# ============================================================
from bed_reader import to_bed

print(f"Simulating {n_snps} SNP loci...")
maf_arr = np.random.uniform(0.05, 0.5, size=10000)

def maf_to_genotype_probs(maf):
    p = 1 - maf
    q = maf
    return np.array([p ** 2, 2 * p * q, q ** 2])

snp_data = np.zeros((n_sample, n_snps), dtype=int)

for i in range(n_snps):
    maf_sample = np.random.choice(maf_arr, size=1, replace=False)[0]
    probs = maf_to_genotype_probs(maf_sample)
    snp_data[:, i] = np.random.choice([0, 1, 2], size=n_sample, p=probs)

snp_data_plink = snp_data
bed_file = os.path.join(output_dir, "sim_genotypes.bed")
bim_file = os.path.join(output_dir, "sim_genotypes.bim")
fam_file = os.path.join(output_dir, "sim_genotypes.fam")

fam_df = pd.DataFrame({
    'FID': np.arange(1, n_sample + 1),
    'IID': np.arange(1, n_sample + 1),
    'Father': 0,
    'Mother': 0,
    'Sex': 0,
    'Phenotype': -9
})
fam_df.to_csv(fam_file, sep=' ', index=False, header=False)

bim_df = pd.DataFrame({
    'Chrom': 1,
    'SNP': [f"rs{idx}" for idx in range(n_snps)],
    'CM': 0,
    'BP': np.arange(1, n_snps + 1),
    'A1': 'A',
    'A2': 'G'
})
bim_df.to_csv(bim_file, sep='\t', index=False, header=False)

to_bed(bed_file, snp_data_plink, properties={"fid": fam_df['FID'].values, "iid": fam_df['IID'].values, "sid": bim_df['SNP'].values})
print(f"PLINK files saved to {output_dir}")

Simulating 100 SNP loci...
PLINK files saved to ../examples/simulation


In [5]:
# ============================================================
# Generate covariates
# ============================================================
n_cov = 2
covariates = np.random.randn(n_sample, n_cov).astype(np.float32)
covariates[:, 0] = np.random.normal(60, 8, n_sample)   # age
covariates[:, 1] = np.random.randint(0, 2, n_sample)   # sex

np.save(os.path.join(output_dir, "covariates.npy"), covariates)
np.save(os.path.join(output_dir, "covariate_ids.npy"), np.arange(1, n_sample + 1)) # Match 1-indexed IDs
print(f"Covariates saved: shape={covariates.shape}")

Covariates saved: shape=(500, 2)


In [6]:
# ============================================================
# Generate imaging data
# ============================================================
def generate_3d_gaussian(shape, center, sigma, amplitude):
    """Generate a 3D Gaussian cluster."""
    x, y, z = np.indices(shape)
    xyz = np.stack([x, y, z], axis=-1)
    distance = np.linalg.norm((xyz - center) / sigma, axis=-1)
    return amplitude * np.exp(-0.5 * distance**2)

gaussian_cluster = generate_3d_gaussian(
    mask_data.shape, center_voxel, sigma=sigma, amplitude=amplitude
)

genotypes = snp_data[:, causal_snp_idx]

signal_mask = np.random.choice(
    [0, 1], size=n_sample, p=[mask_rate, 1 - mask_rate]
)
print(
    f"  Subjects with signal: {np.sum(signal_mask)} / {n_sample} "
    f"({np.sum(signal_mask) / n_sample * 100:.1f}%)"
)

random_data = np.random.randn(n_sample, *mask_data.shape)
gaussian_3d_expanded = gaussian_cluster[np.newaxis, ...]
genotype_weights = genotypes.reshape(-1, 1, 1, 1)
mask_weights = signal_mask.reshape(-1, 1, 1, 1)
final_weights = genotype_weights * mask_weights

simulated_stat = random_data + final_weights * gaussian_3d_expanded
simulated_stat[:, mask_data == 0] = 0

  Subjects with signal: 249 / 500 (49.8%)


In [7]:
# ============================================================
# Save per-subject NIfTI
# ============================================================
print(f"Saving NIfTI for {n_sample} subjects...")
for sidx in range(n_sample):
    subj_img = nib.Nifti1Image(
        simulated_stat[sidx].astype(np.float32),
        affine=mask_affine,
        header=mask_header,
    )
    nib.save(subj_img, os.path.join(subj_dir, f"sub-{sidx + 1:04d}.nii.gz"))
print(f"Subject NIfTIs saved to: {subj_dir}")

voxel_mean = np.mean(simulated_stat, axis=0)
nib.save(
    nib.Nifti1Image(voxel_mean.astype(np.float32), affine=mask_affine, header=mask_header),
    os.path.join(voxel_dir, "mean_image_data.nii.gz"),
)
print("Mean NIfTI saved")

Saving NIfTI for 500 subjects...
Subject NIfTIs saved to: ../examples/simulation/voxel_data/subjects
Mean NIfTI saved


### Data Loading (Load from simulated files)

In [8]:
# ============================================================
# Load image data
# ============================================================
from joblib import Parallel, delayed
def read_single_image(img_path, filetype, mask_bool=None, network_index=0, rh_img_path=None):
    """Reads a single image based on modality type."""
    if filetype == 'npy':
        data = np.load(img_path)[:, network_index].flatten()
    elif filetype == 'mgh':
        lh_data = nib.load(img_path).get_fdata().flatten()
        rh_data = nib.load(rh_img_path).get_fdata().flatten()
        data = np.concatenate((lh_data, rh_data))
    elif filetype == 'nii.gz':
        data = nib.load(img_path).get_fdata()
        if mask_bool is not None:
            data = data[mask_bool]
        else:
            data = data.flatten()
    else:
        raise ValueError(f"Unsupported file type: '{filetype}'.")
    return data

image_paths = sorted(glob.glob(os.path.join(subj_dir, "sub-*.nii.gz")))
imaging_ids = [int(os.path.basename(path).split('-')[1].split('.')[0]) for path in image_paths]
print(f"Loading {len(image_paths)} subjects in parallel...")

results = Parallel(n_jobs=16, prefer="threads")(
    delayed(read_single_image)(img_path, 'nii.gz', mask_bool) 
    for img_path in image_paths
)

imaging_matrix = np.array(results)
imaging_ids = np.array(imaging_ids)
print(f"Loaded imaging matrix shape: {imaging_matrix.shape}")

Loading 500 subjects in parallel...
Loaded imaging matrix shape: (500, 228453)


In [10]:
from pandas_plink import read_plink1_bin
# ============================================================
# Load genotype data
# ============================================================
print("Reading PLINK files...")
G = read_plink1_bin(bed_file, bim_file, fam_file, verbose=False)
genotype_ids = G.iid.values.astype(int)
plink_snps = G.snp.values
genotype_matrix = np.array(G.values)
print(f"Loaded genotype matrix shape: {genotype_matrix.shape}")

Reading PLINK files...
Loaded genotype matrix shape: (500, 100)


In [11]:
# ============================================================
# Load covariate data
# ============================================================
covariates_matrix = np.load(os.path.join(output_dir, "covariates.npy"))
cov_ids = np.load(os.path.join(output_dir, "covariate_ids.npy"))

# ============================================================
# ID Matching
# ============================================================
common_ids = np.intersect1d(imaging_ids, genotype_ids)
common_ids = np.intersect1d(common_ids, cov_ids)
_, idx_img, _ = np.intersect1d(common_ids, imaging_ids, return_indices=True)
_, idx_geno, _ = np.intersect1d(common_ids, genotype_ids, return_indices=True)
_, idx_cov, _ = np.intersect1d(common_ids, cov_ids, return_indices=True)

# Reorder the matrices to match the common IDs
simulated_stat = imaging_matrix[idx_img, :]
snp_data = 2 - np.array(G.values)[idx_geno, :]
covariates = covariates_matrix[idx_cov, :]

print(f"Total matched samples: {len(common_ids)}")

Total matched samples: 500


### BWAS

In [ ]:
from voxelgenetics.bwas.regression import estimate_bwas

print("Running BWAS on simulated data...")

estimate_bwas(
    genotype_bed=snp_data,
    imaging_matrix=imaging_matrix,
    covariates=covariates,     
    residualize_y=True,  
    include_covariates_in_model=False,

    sampleid=common_ids, 
    snplist=plink_snps, 
    
    burden_threshold_ratio=0.01,
    chunk_size=500,
    burden_batch_size=50,
    device='cuda:0',

    save_mode='npy',
    output_prefix=f"{output_dir}/bwas", 
    npy_t_threshold=3.29,
    npy_min_sig_voxels=300,
)
print(f"BWAS completed. Results saved to {output_dir}/bwas/")

### 3D Voxel-level RFT correction

In [17]:
from voxelgenetics.rft.volume import cluster_correct_volume
from voxelgenetics.atlas.loaders import load_atlas_by_name

img = np.load(f'{output_dir}/bwas/snp_{causal_snp_idx}_tstat.npy')
stat = np.zeros(mask_data.shape)
stat[mask_bool] = img

ukb_data, region_dict = load_atlas_by_name("vbm", data_dir='../data/')

df = cluster_correct_volume(
    stat_img=stat,
    mask=mask_bool,
    cdt_z=3.29,
    alpha=5e-8,                # FWE significance threshold
    
    # Optional parameters:
    fwhm=None,                # None → automatically estimate FWHM from `stat_img`
                              # Or provide your own calculated value, e.g., fwhm=8.5 or (7, 7, 9.5)
    affine=info1.affine,        # Provide → returns peak MNI coordinates in the output
    atlas=(ukb_data, region_dict), # Provide → returns brain region overlap in the output
    output_file=f"{output_dir}/bwas/snp_{causal_snp_idx}_corrected_result.nii.gz", # Provide → saves a NIfTI file with only significant clusters
    min_voxel_count=30, # Use `min_voxel_count=100` to only report regions with ≥100 voxels in the cluster.
)
print(df)

Saved 1 significant clusters to ../examples/simulation/bwas/snp_50_corrected_result.nii.gz
   cluster_index  cluster_size  cluster_p  cluster_p_uncorrected  peak_stat  \
0            120            96        0.0          1.370983e-173    7.33638   

     peak_voxel  peak_mni_x  peak_mni_y  peak_mni_z  \
0  (44, 90, 29)         2.0        54.0       -14.0   

                                       brain_regions  
0  {'Left Frontal Medial Cortex': {'region_name':...  


/Users/qinmenghan/Desktop/code_vbwas/main/src/voxelgenetics/rft/volume.py:23: RuntimeWarning: invalid value encountered in divide
  fMRI = fMRI * fMRI / fMRI
